# BigBounce GPU Research Session — RunPod

**Purpose:** Run GPU-intensive models and large-scale astronomical ML on RunPod (RTX 6000 Ada, 48GB VRAM).

**Also works on:** Google Colab Pro (A100), Lambda Cloud, or any local GPU.

**Pod workspace:** `/workspace/` — persistent volume, survives pod restarts.

**Repo:** [Hubify-Projects/bigbounce](https://github.com/Hubify-Projects/bigbounce)

**Setup:** If running on RunPod JupyterLab, the environment should already be configured via `runpod_cloud.py setup`.

In [ ]:
# ── 0. Install dependencies (skip if already installed via setup) ────
!pip install -q transformers datasets astroML astroquery \
    anthropic openai google-generativeai python-dotenv \
    scipy matplotlib seaborn huggingface_hub timm \
    cobaya camb healpy getdist requests pandas

In [ ]:
# ── 1. Detect environment and load API keys ─────────────────────────
import os, sys
from pathlib import Path

# Auto-detect platform
if Path('/workspace').exists():
    PLATFORM = 'runpod'
    REPO = '/workspace/bigbounce'
elif os.environ.get('COLAB_GPU') or Path('/content').exists():
    PLATFORM = 'colab'
    REPO = '/content/bigbounce'
else:
    PLATFORM = 'local'
    REPO = str(Path('.').resolve())

print(f'Platform: {PLATFORM}')
print(f'Repo: {REPO}')

# Load API keys
env_file = Path(REPO) / '.env.local'
if env_file.exists():
    try:
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print(f'Loaded keys from {env_file}')
    except ImportError:
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip())
        print(f'Loaded keys from {env_file} (manual parse)')
elif PLATFORM == 'colab':
    from google.colab import userdata
    for k in ['HUGGINGFACE_TOKEN', 'ANTHROPIC_API_KEY', 'GOOGLE_AI_API_KEY', 'DEEPSEEK_API_KEY']:
        try: os.environ[k] = userdata.get(k)
        except: pass
    print('Loaded keys from Colab Secrets')

# Verify GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({mem:.1f} GB)')
else:
    print('WARNING: No GPU detected.')

In [ ]:
# ── 2. Clone/update BigBounce repo and import agents ────────────────
import subprocess

if not Path(REPO).exists():
    subprocess.run(['git', 'clone', 'https://github.com/Hubify-Projects/bigbounce.git', REPO])
else:
    subprocess.run(['git', '-C', REPO, 'pull'])

if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Import research agents
from research.agents.dataset_loaders import (
    load_mmu, load_astroml, load_polymathic,
    list_mmu_datasets, list_astroml_datasets, list_polymathic_models
)
print('Research agents loaded')
print(f'MMU datasets: {list(list_mmu_datasets().keys())}')
print(f'Polymathic models: {list_polymathic_models()}')

## Polymathic AI Models

Physics foundation models trained on simulation data:
- **Walrus (1.3B):** Simulation-based inference — needs ~4GB VRAM
- **AION-base (0.3B):** Astronomical omnimodal network — fast inference
- **AION-large (3.1B):** Full astronomical omnimodal — needs ~12GB VRAM

RunPod RTX 6000 Ada (48GB) can run all three simultaneously.

In [ ]:
# ── 3. Load Polymathic AI — Walrus (1.3B) ───────────────────────────
from transformers import AutoModel

hf_token = os.environ.get('HUGGINGFACE_TOKEN', '')

print('Loading Walrus (1.3B params)...')
walrus = AutoModel.from_pretrained(
    'polymathic-ai/walrus',
    token=hf_token if hf_token else None,
    device_map='auto',
    trust_remote_code=True,
)
print(f'Walrus loaded on {next(walrus.parameters()).device}')
print(f'Parameters: {sum(p.numel() for p in walrus.parameters()):,}')

## Multimodal Universe Datasets

100TB of astronomical data on HuggingFace. Stream without downloading.

In [ ]:
# ── 4. Stream PLAsTiCC light curves ─────────────────────────────────
plasticc = load_mmu('plasticc', streaming=True, max_samples=100)

sample = next(iter(plasticc))
print(f'PLAsTiCC sample keys: {list(sample.keys())}')
print(f'Target: {sample.get("target", "N/A")}')

In [ ]:
# ── 5. Stream JWST CEERS galaxy images ──────────────────────────────
ceers = load_mmu('jwst_ceers', streaming=True, max_samples=10)

sample = next(iter(ceers))
print(f'JWST CEERS sample keys: {list(sample.keys())}')

In [ ]:
# ── 6. Load SDSS spectra via AstroML ────────────────────────────────
spectra = load_astroml('sdss_spectra')
print(f'SDSS spectra shape: {spectra.shape}')
print(f'Columns: {spectra.dtype.names[:10]}...')

## MAST / JWST Data Access

Query JWST observations directly. Free, no auth needed.

In [ ]:
# ── 7. Query JWST observations ──────────────────────────────────────
from research.agents.data_access import search_jwst, search_gaia

obs = search_jwst('NGC 1365', radius_arcmin=5)
print(f'Found {len(obs)} JWST observations of NGC 1365')
for o in obs[:5]:
    print(f"  {o['obs_id']}: {o['instrument']} {o['filters']} ({o['exposure_time']:.0f}s)")

In [ ]:
# ── 8. Query Gaia DR3 — Galactic Center ─────────────────────────────
stars = search_gaia(ra=266.4, dec=-29.0, radius_deg=0.1, max_results=20)
print(f'Found {len(stars)} Gaia DR3 sources near Galactic Center')
for s in stars[:5]:
    print(f"  Gaia {s['source_id']}: G={s['phot_g_mean_mag']:.2f}, parallax={s['parallax']:.3f} mas")

## BigBounce-Specific Analysis

Templates for research tasks directly relevant to the paper.

In [ ]:
# ── 9. CMB parity analysis data retrieval ────────────────────────────
from research.agents.data_access import search_mast

planck = search_mast(target='Planck', collection='Planck')
print(f'Planck products in MAST: {len(planck)}')

In [ ]:
# ── 10. Cross-check equations with DeepSeek R1 ──────────────────────
from research.agents.computation import deepseek_verify

result = deepseek_verify(
    claim="The inflationary suppression factor D_inf = e^{-3N}(T_reh/M_GUT)^{3/2} "
          "gives D_inf ~ 10^{-121} for N=92, T_reh=10^{15} GeV, M_GUT=10^{16} GeV",
    context="Einstein-Cartan cosmology with Holst term, evaluating parity-odd operator "
            "on-shell during inflation"
)
print(f"Verdict: {result['verdict']}")
print(f"\n{result['content'][:1000]}")

## Save Session Results

Persist outputs to `/workspace/outputs/` (RunPod) or `research/outputs/` (local).

In [ ]:
# ── 11. Save session report ──────────────────────────────────────────
import json
from datetime import datetime

report = {
    'timestamp': datetime.now().isoformat(),
    'platform': PLATFORM,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
    'vram_gb': round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1) if torch.cuda.is_available() else 0,
    'repo': REPO,
}

out_dir = Path('/workspace/outputs') if PLATFORM == 'runpod' else Path(REPO) / 'research' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / f"gpu_session_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
out_file.write_text(json.dumps(report, indent=2))
print(f'Session report saved to {out_file}')